In [0]:
import requests
import json
import time

# 1. Definicja zapytania Overpass QL (Zwiększony timeout serwera do 90s, nieco zawężony obszar)
overpass_query = """
[out:json][timeout:90];
(
  node["harbour"="yes"](35.0, -10.0, 65.0, 30.0);
  node["industrial"="port"](35.0, -10.0, 65.0, 30.0);
);
out body;
"""

url = "http://overpass-api.de/api/interpreter"

# Pobieramy User-Agent (wraz z prawdziwym adresem e-mail) bezpiecznie z Key Vault / Databricks Secrets
api_user_agent = dbutils.secrets.get(scope="maritime_kv", key="osm-user-agent")

headers = {
    "User-Agent": api_user_agent,
    "Accept": "application/json"
}

# Wzorzec Retry (Maksymalnie 3 próby pobrania na wypadek przeciążenia serwerów Overpass API)
max_retries = 3
retry_delay_seconds = 5
osm_data = None

for attempt in range(1, max_retries + 1):
    try:
        print(f"Pobieranie danych z Overpass API (Próba {attempt}/{max_retries})...")
        response = requests.post(url, data=overpass_query, headers=headers, timeout=120)
        response.raise_for_status() 
        
        osm_data = response.json()
        port_count = len(osm_data.get('elements', []))
        print(f"Pobrano pomyślnie dane z OSM. Znaleziono portów: {port_count}")
        break # Sukces, wychodzimy z pętli

    except requests.exceptions.RequestException as e:
        print(f"Błąd podczas próby {attempt}: {e}")
        if attempt < max_retries:
            print(f"Czekam {retry_delay_seconds} sekund przed ponowieniem...")
            time.sleep(retry_delay_seconds)
        else:
            print("Wyczerpano wszystkie próby. API jest obecnie niedostępne.")
            raise e

# 2. Zapis w warstwie Bronze na Data Lake (wykonuje się tylko, jeśli osm_data istnieje)
if osm_data:
    adls_key = dbutils.secrets.get(scope="maritime_kv", key="adls-bronze-key")
    storage_account_name = "adlsmaritimegen2"
    spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", adls_key)

    # Przygotowujemy jedną, wielką ścieżkę tekstową z JSON-em
    json_string = json.dumps(osm_data)
    df_raw_ports = spark.createDataFrame([(json_string,)], ["raw_json"])

    bronze_ports_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/osm_ports_raw/ports_europe.json"

    df_raw_ports.write.mode("overwrite").text(bronze_ports_path)
    print(f"Zapisano surowy plik JSON na Data Lake: {bronze_ports_path}")